# Step 1 — See exactly what the model will receive

In [1]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

lab12 = root / "data" / "processed" / "lab12"
alerts = pd.read_parquet(lab12 / "alert_queries.parquet").set_index("source_row")
passages = pd.read_parquet(lab12 / "guidance_passages.parquet").set_index("passage_id")
rankings = pd.read_csv(root / "reports" / "lab_12_retrieval_comparison.csv")

In [2]:
alert_id = 695464
encoder_rankings = rankings.loc[rankings["method"] == "Original encoder"].set_index("source_row")
passage_ids = encoder_rankings.at[alert_id, "top_three"].split(", ")[:2]

alert_text = alerts.at[alert_id, "query_text"]
evidence = [f"[TX-{alert_id}] {alert_text}"]

for passage_id in passage_ids:
    evidence.append(f"[{passage_id}] {passages.at[passage_id, 'text']}")

context = "\n\n".join(evidence)
print(context)

[TX-695464] The account sent an ACH payment. The amount was 18,533.55 (currency: US Dollar). In the preceding 24 hours, the account received 3 incoming and sent 2 outgoing payments.

[P07] An account receives money and passes it onward to beneficiaries.

[P03] Funds enter and leave the same account over a short interval.


# Step 2 — Generate the first review note

In [4]:
prompt = f"""Write a two-sentence note for a transaction reviewer using only the evidence below.

1. State what the transaction record shows. End the sentence with its [TX-...] ID.
2. Suggest one specific follow-up check. End the sentence with the [P...] ID
   that supports the check.

Treat guidance passages as possible leads. Only the transaction record can
establish what happened in this account.

Evidence:
{context}
"""

In [8]:
import json
from urllib.request import Request, urlopen

payload = {
    "model": "qwen3:4b",
    "prompt": prompt,
    "stream": False,
    "think": False,
    "options": {"temperature": 0, "num_ctx": 2048, "num_predict": 128},
    "keep_alive": 0,
}

payload["prompt"] = prompt + "\nReturn JSON. Put sentence 1 in observation and sentence 2 in follow_up."
payload["format"] = {
    "type": "object",
    "properties": {
        "observation": {"type": "string"},
        "follow_up": {"type": "string"},
    },
    "required": ["observation", "follow_up"],
    "additionalProperties": False,
}

request = Request(
    "http://localhost:11434/api/generate",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)

with urlopen(request, timeout=180) as response:
    result = json.load(response)

note = json.loads(result["response"])
review_note = f"{note['observation']}\n{note['follow_up']}"
print(review_note)

The transaction record shows that the account sent an ACH payment of 18,533.55 USD with TX-695464.
Check if the account received funds within the last 24 hours using P03.


### Evaluation. The model summarized the payment accurately. Its follow-up asked a question the alert already answers. A more useful check would be “Compare the timestamps of the three incoming payments with the outgoing ACH to see how close together they occurred [P03].” The model included both source IDs but omitted the requested brackets.

# OpenAI Model

In [9]:
import os
from dotenv import load_dotenv
from getpass import getpass

from openai import OpenAI

load_dotenv(root / ".env")
client = OpenAI()

In [16]:
import json

response = client.responses.create(
    model="gpt-6-luna",
    reasoning={"effort": "none"},
    input=payload["prompt"],
    text={
        "format": {
            "type": "json_schema",
            "name": "review_note",
            "strict": True,
            "schema": payload["format"],
        }
    },
    max_output_tokens=128,
    store=False,
)

if response.status != "completed" or not response.output_text:
    raise RuntimeError(f"No complete note returned. Details: {response.incomplete_details}")

openai_note = json.loads(response.output_text)
print(openai_note["observation"])
print(openai_note["follow_up"])

The account sent an ACH payment of $18,533.55 after receiving 3 incoming payments and sending 2 outgoing payments in the preceding 24 hours [TX-695464].
Check whether the recent incoming payments were passed onward to beneficiaries [P07].


In [17]:
usage = response.usage
cost_usd = (usage.input_tokens * 0.10 + usage.output_tokens * 0.50) / 1_000_000

print(f"{usage.input_tokens} input tokens, {usage.output_tokens} output tokens")
print(f"Estimated cost: ${cost_usd:.6f} USD")

215 input tokens, 69 output tokens
Estimated cost: $0.000056 USD
